# 05. 지도 기반 시각화

이상 탐지 결과를 지도 위에 시각화한다.

- 정상 항적(blue) vs 이상 항적(red) 오버레이
- 이상 선박 개별 항적 추적
- 이상 유형별 공간 분포

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap, MarkerCluster
import sys
sys.path.append('..')

from src.visualize import plot_trajectory_map, plot_anomaly_distribution

In [ ]:
df = pd.read_parquet('../data/ais_results.parquet')
print(f'{len(df):,} rows')
print(f'정상: {(df["anomaly_final"] == 0).sum():,}')
print(f'이상: {(df["anomaly_final"] == 1).sum():,}')

## 5.1 정상 vs 이상 항적 지도

In [ ]:
m = plot_trajectory_map(df)
m.save('../results/figures/anomaly_map.html')
print('저장: results/figures/anomaly_map.html')
m

## 5.2 이상 선박 Top 10 항적 추적

In [ ]:
# 이상 레코드가 가장 많은 선박 Top 10
anomaly_df = df[df['anomaly_final'] == 1]
top_anomaly_vessels = anomaly_df['MMSI'].value_counts().head(10)
print('이상 레코드 상위 10척:')
print(top_anomaly_vessels)

# 상위 3척 항적 시각화
colors = ['red', 'orange', 'yellow']
center = [df['LAT'].mean(), df['LON'].mean()]
m = folium.Map(location=center, zoom_start=5, tiles='CartoDB dark_matter')

for i, mmsi in enumerate(top_anomaly_vessels.index[:3]):
    vessel_data = df[df['MMSI'] == mmsi].sort_values('BaseDateTime')
    coords = vessel_data[['LAT', 'LON']].values.tolist()
    
    # 전체 항적 (선)
    folium.PolyLine(coords, color=colors[i], weight=2, opacity=0.7,
                    tooltip=f'MMSI: {mmsi}').add_to(m)
    
    # 이상 구간 (마커)
    anomaly_points = vessel_data[vessel_data['anomaly_final'] == 1]
    for _, row in anomaly_points.iterrows():
        folium.CircleMarker(
            location=[row['LAT'], row['LON']],
            radius=5, color=colors[i], fill=True, opacity=0.9,
            tooltip=f'MMSI: {mmsi}<br>SOG: {row["SOG"]}<br>Time: {row["BaseDateTime"]}'
        ).add_to(m)

m.save('../results/figures/top_anomaly_vessels.html')
print('\n저장: results/figures/top_anomaly_vessels.html')
m

## 5.3 이상 분포 차트

In [ ]:
fig = plot_anomaly_distribution(df, save_path='../results/figures/anomaly_distribution.png')
plt.show()

## 5.4 이상 유형별 히트맵

In [ ]:
# 속도 이상 vs 침로 이상 vs 신호 단절 공간 분포
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

anomaly_data = df[df['anomaly_final'] == 1]

# 속도 이상
speed_anom = anomaly_data[anomaly_data['speed_deviation'].abs() > 3]
axes[0].scatter(speed_anom['LON'], speed_anom['LAT'], s=1, c='red', alpha=0.3)
axes[0].set_title(f'Speed Anomaly ({len(speed_anom):,})')
axes[0].set_xlabel('LON')
axes[0].set_ylabel('LAT')

# 침로 급변
course_anom = anomaly_data[anomaly_data['course_change'] > 90]
axes[1].scatter(course_anom['LON'], course_anom['LAT'], s=1, c='orange', alpha=0.3)
axes[1].set_title(f'Course Change > 90° ({len(course_anom):,})')
axes[1].set_xlabel('LON')

# 신호 단절
signal_anom = anomaly_data[anomaly_data['signal_gap_sec'] > 3600]
axes[2].scatter(signal_anom['LON'], signal_anom['LAT'], s=1, c='yellow', alpha=0.3)
axes[2].set_title(f'Signal Gap > 1hr ({len(signal_anom):,})')
axes[2].set_xlabel('LON')

plt.tight_layout()
plt.savefig('../results/figures/anomaly_types_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5 시간대별 이상 빈도

In [ ]:
df['hour'] = df['BaseDateTime'].dt.hour

fig, ax = plt.subplots(figsize=(12, 5))

hourly_total = df.groupby('hour').size()
hourly_anomaly = df[df['anomaly_final'] == 1].groupby('hour').size()
hourly_rate = (hourly_anomaly / hourly_total * 100).fillna(0)

ax.bar(hourly_rate.index, hourly_rate.values, color='coral', alpha=0.8)
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Anomaly Rate (%)')
ax.set_title('Anomaly Rate by Hour')
ax.set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('../results/figures/hourly_anomaly_rate.png', dpi=150, bbox_inches='tight')
plt.show()